# HW2 Preparation: Single-shot generation vs. self-consistency with LangChain

This notebook runs the same evaluator with two solver files:

- `solver_single.py`, one deterministic generation.
- `solver_self_consistency.py`, five stochastic generations and majority vote.

## 수강생 가이드

이번 과제에서는 내부적으로 LLM을 잘 호출하여 주어진 문제 instance (`example`)에 대한 답을 정하여 리턴하는 `predict(example)` 함수를 작성하시고, 해당 함수를 포함한 solver script (`.py` 파일)를 제출하시면 됩니다.

**작업 루프**
1. `solver_mine.py` 를 편집 (아래 6.5절에서 자동 생성됨)
2. `--limit 2` 로 빠르게 돌려 동작을 확인
3. Dev set (26문항) 으로 정확도, 레이턴시 등을 확인
4. 결과 CSV에서 틀린 문항을 분석 → 1부터 반복

**확인하시면 좋은 사항**
- ✅ **`solver_mine.py` 만 편집하시면 됩니다.** `solver_single.py` / `solver_self_consistency.py` 는 *예시*이고, 아래 "파일 쓰기" 셀을 다시 실행하면 덮어쓰여집니다.
- ✅ 빠른 반복엔 `--limit n`, 전체 26문항 평가는 마지막에.
- ✅ 모든 LLM 호출은 colab에서 돌아가는 로컬 Ollama 서버가 처리하도록 되어있습니다. 제출된 파일은 저희 서버에 똑같은 Qwen 모델을 올려 평가할 예정입니다.

**채점 contract**
`predict(example)` 는 다음을 입력으로 받고 — `example["question"]`, `example["options"]`(리스트), `example["category"]` — 아래 중 하나를 반환합니다.
- 간단형: 정답 문자 하나, 예: `"A"`
- 확장형: `{"answer": "A", "raw": "<모델 원문>", "artifacts": {<디버그용 자유 JSON>}}`


In [1]:
# Optional but recommended: use GPU runtime.
# Runtime > Change runtime type > T4 GPU

!nvidia-smi || true

Wed Jul  1 00:33:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install Python packages

In [2]:
!pip -q install langchain-core langchain-ollama datasets pandas tqdm

In [3]:
!apt-get update -qq
!apt-get install -y zstd pciutils lshw

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 108 not upgraded.
Need to get 1,486 kB of archives.
After this operation, 4,951 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu ja

## 2. Mount Google Drive and choose a work directory

In [4]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Change the path as needed
WORK_DIR = Path('/content/drive/MyDrive/nlp-hw2/')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR =", WORK_DIR)
print("Files currently in WORK_DIR:")
for p in sorted(WORK_DIR.glob('*')):
    print(" -", p.name)

Mounted at /content/drive
WORK_DIR = /content/drive/MyDrive/nlp-hw2
Files currently in WORK_DIR:


## 3. Start Ollama

In [5]:
import subprocess
import time

ollama_log = open("/tmp/ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=ollama_log,
)

time.sleep(8)
print("Ollama server started.")

Ollama server started.


## 4. Pull Qwen3 0.6B

In [6]:
!ollama pull qwen3:0.6b

## 5. Smoke test

In [27]:
!ollama run --think=false qwen3:0.6b "Return only the letter A."

A.



In [28]:
!ollama run --think=true qwen3:0.6b "Return only the letter A."

Thinking...
Okay, the user wants me to return only the letter A. Let me start by unders
understanding the request. They specified only the letter A, which probably
probably means they want the output to be that single character, not any ot
other characters. 

First, I need to make sure I'm not making any mistakes here. If there's any
any other character they want included, but the query says to return only A
A, then I should stick to that. Maybe they are testing if I can respond cor
correctly without adding extra text. 

I should check if there's any ambiguity in the query. The user wrote "Retur
"Return only the letter A." which is straightforward. There's no mention of
of any other requirements, so my job is clear. 

Now, I need to present the answer as a single character. Since the user did
didn't provide any context or additional information, the response should j
just be the letter A. There's no need to include other characters. 

I think that's all. Let me just confirm once more: 

## 6. Write evaluator and solver files

In [8]:
from pathlib import Path

# 최초 1회만 파일을 쓰고, 이후 재실행 시 자동으로 보존(덮어쓰지 않음)합니다.
# 학생이 편집한 솔버가 사라지는 것을 막기 위함입니다.
# 채점기/베이스라인을 강제로 다시 받으려면 FORCE_OVERWRITE = True 로 한 번 실행하세요.
FORCE_OVERWRITE = False
_marker = WORK_DIR / '.runner_written'
OVERWRITE_RUNNER_FILES = FORCE_OVERWRITE or (not _marker.exists())
print('OVERWRITE_RUNNER_FILES =', OVERWRITE_RUNNER_FILES,
      '(first run)' if not _marker.exists() else '(marker exists -> 보존 모드)')

files_to_write = {
    "eval.py": '#!/usr/bin/env python3\n\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport pandas as pd\nfrom datasets import load_dataset\nfrom tqdm import tqdm\n\n\nLETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"\n\n\ndef load_submission_module(path: str):\n    path_obj = Path(path)\n    if not path_obj.exists():\n        raise FileNotFoundError(f"Solver file not found: {path}")\n\n    spec = importlib.util.spec_from_file_location("student_solver", str(path_obj))\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Failed to import solver: {path}")\n\n    module = importlib.util.module_from_spec(spec)\n    sys.modules["student_solver"] = module\n    spec.loader.exec_module(module)\n\n    if not hasattr(module, "predict"):\n        raise RuntimeError("Solver file must define predict(example).")\n\n    return module\n\n\ndef maybe_parse_list(value: Any) -> List[str]:\n    if isinstance(value, list):\n        return [str(x) for x in value]\n\n    if isinstance(value, str):\n        text = value.strip()\n        if text.startswith("[") and text.endswith("]"):\n            try:\n                parsed = json.loads(text)\n                if isinstance(parsed, list):\n                    return [str(x) for x in parsed]\n            except Exception:\n                pass\n\n        return [x.strip() for x in text.split("\\n") if x.strip()]\n\n    raise ValueError(f"Cannot parse options: {value!r}")\n\n\ndef get_options(example: Dict[str, Any]) -> List[str]:\n    for key in ["options", "choices", "answer_choices"]:\n        if key in example:\n            return maybe_parse_list(example[key])\n    raise KeyError(f"Could not find options field in example keys: {list(example.keys())}")\n\n\ndef get_question(example: Dict[str, Any]) -> str:\n    for key in ["question", "input", "query"]:\n        if key in example:\n            return str(example[key])\n    raise KeyError(f"Could not find question field in example keys: {list(example.keys())}")\n\n\ndef get_gold_label(example: Dict[str, Any], options: List[str]) -> str:\n    for key in ["answer", "gold", "label", "target"]:\n        if key not in example:\n            continue\n\n        value = example[key]\n\n        if isinstance(value, str):\n            text = value.strip()\n            if len(text) == 1 and text.upper() in LETTERS[: len(options)]:\n                return text.upper()\n\n            for i, opt in enumerate(options):\n                if text == str(opt).strip():\n                    return LETTERS[i]\n\n        if isinstance(value, int):\n            if 0 <= value < len(options):\n                return LETTERS[value]\n            if 1 <= value <= len(options):\n                return LETTERS[value - 1]\n\n    raise KeyError(f"Could not parse gold label from example: {example}")\n\n\ndef normalize_pred(value: Any, n_options: int) -> Optional[str]:\n    valid = set(LETTERS[:n_options])\n    text = str(value or "").strip().upper()\n\n    if len(text) == 1 and text in valid:\n        return text\n\n    for marker in ["FINAL ANSWER:", "ANSWER:", "OPTION:"]:\n        if marker in text:\n            tail = text.split(marker, 1)[1].strip()\n            if tail and tail[0] in valid:\n                return tail[0]\n\n    for token in text.replace(".", " ").replace(")", " ").replace(":", " ").split():\n        if token in valid:\n            return token\n\n    for ch in text:\n        if ch in valid:\n            return ch\n\n    return None\n\n\ndef safe_json_dumps(value: Any) -> str:\n    try:\n        return json.dumps(value, ensure_ascii=False)\n    except Exception:\n        return json.dumps(str(value), ensure_ascii=False)\n\n\ndef normalize_solver_output(output: Any, n_options: int) -> Dict[str, Any]:\n    """\n    Generic solver output contract.\n\n    Simple form:\n        predict(example) -> "A"\n\n    Rich form:\n        predict(example) -> {\n            "answer": "A",\n            "raw": "... representative raw output ...",\n            "artifacts": {... arbitrary solver-specific JSON ...}\n        }\n\n    eval.py does not know about self-consistency, RAG, tools, etc.\n    It only saves artifacts as opaque JSON.\n    """\n    if isinstance(output, dict):\n        answer = output.get("answer", output.get("pred", None))\n\n        raw = (\n            output.get("raw", None)\n            or output.get("raw_generation", None)\n            or output.get("raw_pred", None)\n            or output.get("response", None)\n            or ""\n        )\n\n        if answer is None or str(answer).strip() == "":\n            pred = normalize_pred(raw, n_options)\n        else:\n            pred = normalize_pred(answer, n_options)\n\n        artifacts = output.get("artifacts", {})\n\n        reserved = {\n            "answer",\n            "pred",\n            "raw",\n            "raw_generation",\n            "raw_pred",\n            "response",\n            "artifacts",\n        }\n        extra = {k: v for k, v in output.items() if k not in reserved}\n\n        if extra:\n            if isinstance(artifacts, dict):\n                artifacts = {**artifacts, "_extra": extra}\n            else:\n                artifacts = {"value": artifacts, "_extra": extra}\n\n        return {\n            "pred": pred,\n            "raw_pred": raw,\n            "artifact_json": safe_json_dumps(artifacts),\n        }\n\n    return {\n        "pred": normalize_pred(output, n_options),\n        "raw_pred": "" if output is None else str(output),\n        "artifact_json": "",\n    }\n\n\ndef parse_csv_list(value: str) -> List[str]:\n    return [x.strip() for x in str(value or "").split(",") if x.strip()]\n\n\ndef build_category_stratified_subset(\n    ds,\n    n_per_category: int = 2,\n    max_categories: int = 0,\n    category_key: str = "category",\n    exclude_categories: Optional[List[str]] = None,\n    include_categories: Optional[List[str]] = None,\n    seed: int = 20260630,\n) -> Tuple[List[int], Dict[str, int]]:\n    exclude_set = {str(x).strip().lower() for x in (exclude_categories or []) if str(x).strip()}\n    include_set = {str(x).strip().lower() for x in (include_categories or []) if str(x).strip()}\n\n    by_category: Dict[str, List[int]] = {}\n\n    for idx, ex in enumerate(ds):\n        category = str(ex.get(category_key, "")).strip()\n        if not category:\n            category = "unknown"\n\n        category_l = category.lower()\n\n        if category_l in exclude_set:\n            continue\n\n        if include_set and category_l not in include_set:\n            continue\n\n        by_category.setdefault(category, []).append(idx)\n\n    categories = sorted(by_category.keys())\n    if max_categories and max_categories > 0:\n        categories = categories[:max_categories]\n\n    rng = random.Random(seed)\n    selected_indices: List[int] = []\n    category_counts: Dict[str, int] = {}\n\n    for category in categories:\n        indices = list(by_category[category])\n        rng.shuffle(indices)\n        chosen = indices[:n_per_category]\n        selected_indices.extend(chosen)\n        category_counts[category] = len(chosen)\n\n    return selected_indices, category_counts\n\n\ndef make_public_example(raw_example: Dict[str, Any], dataset_idx: int) -> Dict[str, Any]:\n    options = get_options(raw_example)\n\n    return {\n        "dataset_idx": dataset_idx,\n        "question": get_question(raw_example),\n        "options": options,\n        "category": str(raw_example.get("category", "")),\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--submission", required=True, help="Path to solver file. The file must define predict(example).")\n    parser.add_argument("--dataset", default="TIGER-Lab/MMLU-Pro")\n    parser.add_argument("--split", default="test")\n    parser.add_argument("--n-per-category", type=int, default=2)\n    parser.add_argument("--max-categories", type=int, default=0)\n    parser.add_argument("--exclude-categories", default="other")\n    parser.add_argument("--include-categories", default="")\n    parser.add_argument("--sample-seed", type=int, default=20260630)\n    parser.add_argument("--output", default="results.csv")\n    parser.add_argument("--summary-output", default="summary.json")\n    parser.add_argument("--limit", type=int, default=0, help="Optional quick debug limit after sampling.")\n    args = parser.parse_args()\n\n    exclude_categories = parse_csv_list(args.exclude_categories)\n    include_categories = parse_csv_list(args.include_categories)\n\n    module = load_submission_module(args.submission)\n\n    print(f"[eval] loading dataset: {args.dataset}/{args.split}", flush=True)\n    ds = load_dataset(args.dataset, split=args.split)\n\n    selected_indices, category_counts = build_category_stratified_subset(\n        ds,\n        n_per_category=args.n_per_category,\n        max_categories=args.max_categories,\n        exclude_categories=exclude_categories,\n        include_categories=include_categories,\n        seed=args.sample_seed,\n    )\n\n    if args.limit and args.limit > 0:\n        selected_indices = selected_indices[: args.limit]\n\n    print(f"[eval] selected {len(selected_indices)} examples", flush=True)\n    print(f"[eval] category_counts={category_counts}", flush=True)\n\n    rows = []\n    correct_count = 0\n    invalid_count = 0\n    total_elapsed = 0.0\n\n    for local_idx, dataset_idx in enumerate(tqdm(selected_indices)):\n        raw_example = dict(ds[dataset_idx])\n        options = get_options(raw_example)\n        gold = get_gold_label(raw_example, options)\n        public_example = make_public_example(raw_example, dataset_idx)\n\n        error = ""\n        raw_pred = ""\n        artifact_json = ""\n        pred = None\n        correct = False\n\n        t0 = time.time()\n        try:\n            solver_output = module.predict(public_example)\n            parsed = normalize_solver_output(solver_output, len(options))\n\n            pred = parsed["pred"]\n            raw_pred = parsed["raw_pred"]\n            artifact_json = parsed["artifact_json"]\n\n            if pred is None:\n                invalid_count += 1\n\n            correct = pred == gold\n            if correct:\n                correct_count += 1\n\n        except Exception as e:\n            error = "".join(traceback.format_exception_only(type(e), e)).strip()[:1000]\n            invalid_count += 1\n\n        elapsed = time.time() - t0\n        total_elapsed += elapsed\n\n        rows.append(\n            {\n                "local_idx": local_idx,\n                "dataset_idx": dataset_idx,\n                "category": public_example["category"],\n                "pred": "" if pred is None else pred,\n                "gold": gold,\n                "correct": bool(correct),\n                "elapsed_sec": elapsed,\n                "raw_pred": "" if raw_pred is None else str(raw_pred),\n                "artifact_json": artifact_json,\n                "error": error,\n                "question": public_example["question"],\n            }\n        )\n\n    n = len(selected_indices)\n    accuracy = correct_count / n if n else 0.0\n    score = round(accuracy * 100, 2)\n\n    summary = {\n        "n": n,\n        "correct": correct_count,\n        "accuracy": accuracy,\n        "score": score,\n        "invalid": invalid_count,\n        "avg_elapsed_sec": total_elapsed / n if n else 0.0,\n        "dataset": args.dataset,\n        "split": args.split,\n        "n_per_category": args.n_per_category,\n        "max_categories": args.max_categories,\n        "exclude_categories": exclude_categories,\n        "include_categories": include_categories,\n        "sample_seed": args.sample_seed,\n        "selected_indices": selected_indices,\n        "category_counts": category_counts,\n    }\n\n    pd.DataFrame(rows).to_csv(args.output, index=False)\n    Path(args.summary_output).write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n\n    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    "solver_single.py": 'from langchain_core.prompts import ChatPromptTemplate\nfrom langchain_ollama import ChatOllama\n\n\nLETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"\n\n\ndef make_llm(\n    temperature=0.0,\n    max_tokens=32,\n    seed=0,\n):\n    return ChatOllama(\n        base_url="http://127.0.0.1:11434",\n        model="qwen3:0.6b",\n        temperature=temperature,\n        num_predict=max_tokens,\n        seed=seed,\n    )\n\n\ndef extract_letter(text, n_options):\n    valid = set(LETTERS[:n_options])\n    text = str(text or "").strip().upper()\n\n    if len(text) == 1 and text in valid:\n        return text\n\n    for marker in ["FINAL ANSWER:", "ANSWER:", "OPTION:"]:\n        if marker in text:\n            tail = text.split(marker, 1)[1].strip()\n            if tail and tail[0] in valid:\n                return tail[0]\n\n    for token in text.replace(".", " ").replace(")", " ").replace(":", " ").split():\n        if token in valid:\n            return token\n\n    for ch in text:\n        if ch in valid:\n            return ch\n\n    return None\n\n\ndef classify_status(raw_response, pred, done_reason):\n    raw = str(raw_response or "")\n\n    if pred is not None:\n        return "ok"\n\n    if not raw.strip():\n        return "empty_response"\n\n    done_reason = str(done_reason or "").lower()\n\n    if done_reason in {"length", "max_tokens", "num_predict"}:\n        return "truncated_no_answer"\n\n    return "no_letter_extracted"\n\n\ndef predict(example):\n    """\n    Single-call baseline.\n\n    Return contract:\n    {\n        "answer": final extracted letter,\n        "raw": full model response,\n        "artifacts": solver-specific debug info\n    }\n    """\n    question = example["question"]\n    options = example["options"]\n    category = example.get("category", "")\n\n    option_text = "\\n".join(\n        f"{LETTERS[i]}. {option}"\n        for i, option in enumerate(options)\n    )\n\n    prompt = ChatPromptTemplate.from_messages(\n        [\n            (\n                "system",\n                "You are a careful multiple-choice solver. Return only one answer letter.",\n            ),\n            (\n                "human",\n                """/no_think\nSolve the following multiple-choice question.\n\nCategory:\n{category}\n\nQuestion:\n{question}\n\nOptions:\n{options}\n\nReturn only the single best answer letter. Do not include any explanation.""",\n            ),\n        ]\n    )\n\n    max_tokens = 4096\n    temperature = 0.0\n    seed = 0\n\n    llm = make_llm(\n        temperature=temperature,\n        max_tokens=max_tokens,\n        seed=seed,\n    )\n\n    chain = prompt | llm\n\n    try:\n        msg = chain.invoke(\n            {\n                "category": category,\n                "question": question,\n                "options": option_text,\n            }\n        )\n\n        raw_response = getattr(msg, "content", str(msg))\n        response_metadata = getattr(msg, "response_metadata", {}) or {}\n        done_reason = response_metadata.get("done_reason", "")\n        exception = ""\n\n    except Exception as e:\n        raw_response = ""\n        response_metadata = {"error": repr(e)}\n        done_reason = "exception"\n        exception = repr(e)\n\n    pred = extract_letter(raw_response, len(options))\n    status = classify_status(raw_response, pred, done_reason)\n\n    return {\n        "answer": pred,\n        "raw": raw_response,\n        "artifacts": {\n            "mode": "single",\n            "model": "qwen3:0.6b",\n            "max_tokens": max_tokens,\n            "temperature": temperature,\n            "seed": seed,\n            "status": status,\n            "done_reason": done_reason,\n            "response_metadata": response_metadata,\n            "exception": exception,\n        },\n    }\n',
    "solver_self_consistency.py": 'from collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\n\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_ollama import ChatOllama\n\n\nLETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"\n\n\ndef make_llm(\n    temperature=0.0,\n    max_tokens=32,\n    seed=0,\n):\n    return ChatOllama(\n        base_url="http://127.0.0.1:11434",\n        model="qwen3:0.6b",\n        temperature=temperature,\n        num_predict=max_tokens,\n        seed=seed,\n    )\n\n\ndef extract_letter(text, n_options):\n    valid = set(LETTERS[:n_options])\n    text = str(text or "").strip().upper()\n\n    if len(text) == 1 and text in valid:\n        return text\n\n    for marker in ["FINAL ANSWER:", "ANSWER:", "OPTION:"]:\n        if marker in text:\n            tail = text.split(marker, 1)[1].strip()\n            if tail and tail[0] in valid:\n                return tail[0]\n\n    for token in text.replace(".", " ").replace(")", " ").replace(":", " ").split():\n        if token in valid:\n            return token\n\n    for ch in text:\n        if ch in valid:\n            return ch\n\n    return None\n\n\ndef classify_status(raw_response, pred, done_reason):\n    raw = str(raw_response or "")\n\n    if pred is not None:\n        return "ok"\n\n    if not raw.strip():\n        return "empty_response"\n\n    done_reason = str(done_reason or "").lower()\n\n    if done_reason in {"length", "max_tokens", "num_predict"}:\n        return "truncated_no_answer"\n\n    return "no_letter_extracted"\n\n\ndef majority_vote(preds, n_options):\n    counts = Counter(preds)\n    max_count = max(counts.values())\n    winners = [p for p, c in counts.items() if c == max_count]\n\n    for letter in LETTERS[:n_options]:\n        if letter in winners:\n            return letter\n\n    return winners[0]\n\n\ndef predict(example):\n    """\n    Self-consistency baseline.\n\n    eval.py only sees the generic contract:\n    {\n        "answer": final majority-vote letter,\n        "raw": human-readable combined raw generations,\n        "artifacts": solver-specific debug info\n    }\n    """\n    return predict_self_consistency(\n        example,\n        n_samples=5,\n        max_workers=3,\n    )\n\n\ndef predict_self_consistency(example, n_samples=5, max_workers=3):\n    question = example["question"]\n    options = example["options"]\n    category = example.get("category", "")\n\n    option_text = "\\n".join(\n        f"{LETTERS[i]}. {option}"\n        for i, option in enumerate(options)\n    )\n\n    prompt_variants = [\n        "Solve directly and choose the best answer.",\n        "Eliminate clearly wrong options first, then choose.",\n        "Check each option briefly and choose the most defensible answer.",\n        "Look for traps or misleading options before choosing.",\n        "Use concise domain reasoning and avoid overthinking.",\n    ]\n\n    prompt = ChatPromptTemplate.from_messages(\n        [\n            (\n                "system",\n                "You are a careful multiple-choice solver. Think briefly, then return the final answer letter.",\n            ),\n            (\n                "human",\n                """/think\n{strategy}\n\nUse very brief reasoning only.\nThink for at most 2 short steps.\nThen return the final answer in this format:\nFinal Answer: <single letter>\n\nCategory:\n{category}\n\nQuestion:\n{question}\n\nOptions:\n{options}""",\n            ),\n        ]\n    )\n\n    max_tokens = 4096\n    temperature = 0.8\n\n    def run_one(i):\n        llm = make_llm(\n            temperature=temperature,\n            max_tokens=max_tokens,\n            seed=i,\n        )\n\n        chain = prompt | llm\n\n        try:\n            msg = chain.invoke(\n                {\n                    "strategy": prompt_variants[i % len(prompt_variants)],\n                    "category": category,\n                    "question": question,\n                    "options": option_text,\n                }\n            )\n\n            raw_response = getattr(msg, "content", str(msg))\n            response_metadata = getattr(msg, "response_metadata", {}) or {}\n            done_reason = response_metadata.get("done_reason", "")\n            exception = ""\n\n        except Exception as e:\n            raw_response = ""\n            response_metadata = {"error": repr(e)}\n            done_reason = "exception"\n            exception = repr(e)\n\n        letter = extract_letter(raw_response, len(options))\n        status = classify_status(raw_response, letter, done_reason)\n\n        return {\n            "sample_idx": i,\n            "seed": i,\n            "strategy": prompt_variants[i % len(prompt_variants)],\n            "letter": letter,\n            "status": status,\n            "done_reason": done_reason,\n            "raw_generation": raw_response,\n            "response_metadata": response_metadata,\n            "exception": exception,\n        }\n\n    results = []\n\n    with ThreadPoolExecutor(max_workers=max_workers) as ex:\n        futures = [ex.submit(run_one, i) for i in range(n_samples)]\n\n        for fut in as_completed(futures):\n            try:\n                results.append(fut.result())\n            except Exception as e:\n                results.append(\n                    {\n                        "sample_idx": None,\n                        "seed": None,\n                        "strategy": "",\n                        "letter": None,\n                        "status": "exception",\n                        "done_reason": "exception",\n                        "raw_generation": "",\n                        "response_metadata": {"error": repr(e)},\n                        "exception": repr(e),\n                    }\n                )\n\n    # Keep sample order stable in artifacts.\n    results = sorted(\n        results,\n        key=lambda x: 10**9 if x.get("sample_idx") is None else x.get("sample_idx"),\n    )\n\n    letters = [\n        result["letter"]\n        for result in results\n        if result.get("letter") is not None\n    ]\n\n    votes = dict(Counter(letters))\n\n    if not letters:\n        pred = None\n        status = "no_valid_samples"\n    else:\n        pred = majority_vote(letters, len(options))\n        status = "ok"\n\n    raw_summary = "\\n\\n".join(\n        "--- SAMPLE {i} | seed={seed} | letter={letter} | status={status} | done_reason={done_reason} ---\\n{raw}".format(\n            i=i,\n            seed=result.get("seed", ""),\n            letter=result.get("letter", ""),\n            status=result.get("status", ""),\n            done_reason=result.get("done_reason", ""),\n            raw=result.get("raw_generation", ""),\n        )\n        for i, result in enumerate(results)\n    )\n\n    return {\n        "answer": pred,\n        "raw": raw_summary,\n        "artifacts": {\n            "mode": "self_consistency",\n            "model": "qwen3:0.6b",\n            "n_samples": n_samples,\n            "max_workers": max_workers,\n            "max_tokens": max_tokens,\n            "temperature": temperature,\n            "votes": votes,\n            "final_answer": pred,\n            "status": status,\n            "samples": results,\n        },\n    }\n',
}

for filename, content in files_to_write.items():
    path = WORK_DIR / filename
    if OVERWRITE_RUNNER_FILES or not path.exists():
        path.write_text(content, encoding="utf-8")
        print("wrote", path)
    else:
        print("kept existing", path)

_marker.touch()  # 최초 작성 표시 -> 다음 실행부터 자동 보존

print("\nRunner files:")
for p in ["eval.py", "solver_single.py", "solver_self_consistency.py"]:
    print(" -", WORK_DIR / p)

OVERWRITE_RUNNER_FILES = True (first run)
wrote /content/drive/MyDrive/nlp-hw2/eval.py
wrote /content/drive/MyDrive/nlp-hw2/solver_single.py
wrote /content/drive/MyDrive/nlp-hw2/solver_self_consistency.py

Runner files:
 - /content/drive/MyDrive/nlp-hw2/eval.py
 - /content/drive/MyDrive/nlp-hw2/solver_single.py
 - /content/drive/MyDrive/nlp-hw2/solver_self_consistency.py


### ⬆️ 위 셀이 만든 파일들
- `eval.py` — 채점기. **수정하지 마세요.**
- `solver_single.py`, `solver_self_consistency.py` — 참고용 베이스라인 *예시*.

> ⚠️ **중요한 함정:** 위 "파일 쓰기" 셀은 `OVERWRITE_RUNNER_FILES=True` 라서 실행할 때마다 위 세 파일을 **다시 씁니다.** 베이스라인 파일을 직접 고치면 다음 실행 때 사라집니다.
> 그래서 아래에서 **`solver_mine.py`** 를 따로 만들고, 여러분은 그 파일만 편집합니다. (이 파일은 위 셀이 건드리지 않습니다.)

## 6.5 솔버 커스터마이즈 가이드 — 어디를 바꾸나

`solver_mine.py` 를 열고 아래 지점을 바꿔 실험하세요. 계약(=`predict` 반환 형식)만 지키면 내부는 자유입니다.

| 바꿀 곳 | 무엇 | 힌트 |
|---|---|---|
| `make_llm()` | `model`, `temperature`, `num_predict`, `seed` | 0.6B는 작아 추론이 약함. `temperature=0` 은 결정적. |
| 프롬프트 첫 줄 `/no_think` ↔ `/think` | thinking 모드 | **정확도·속도의 가장 큰 레버.** `/think` 는 정확↑·느림↑(그래서 `num_predict` 를 크게). `/no_think` 는 빠름. |
| system / human 프롬프트 | 지시·출력 형식 | `Final Answer: <letter>` 처럼 형식을 강제하면 파싱이 안정됩니다. |
| `extract_letter()` | 답 파싱 | 모델이 형식을 벗어나도 글자를 건지도록 보강. |
| (self-consistency) 샘플 수·temperature·전략 | 다양성 | 표가 갈리면 샘플↑. 단 호출수↑ → 느려짐. |

**추천 진행**: vanilla → 프롬프트 CoT(`/think`) → self-consistency → (다음 안내의) reflexion. 변경할 때마다 `--limit 2` 로 빠르게 확인하고, 그다음 전체로 측정하세요.

아래 셀을 실행하면 주석이 달린 시작 템플릿 `solver_template.py` 가 생기고, 편집용 사본 `solver_mine.py` 가 (없을 때만) 만들어집니다.

In [9]:
from pathlib import Path
# WORK_DIR 는 위 '2. Mount' 셀에서 정의됨
solver_template = r'''
# =====================================================================
# solver_mine.py  —  여러분이 편집하는 솔버 (이 파일만 고치세요)
#
# 채점 계약 (eval.py 가 호출):
#   predict(example) -> "A"   (정답 문자 하나)
#               또는  -> {"answer": "A", "raw": "...", "artifacts": {...}}
#   example = {"question": str, "options": [str, ...], "category": str}
#   * 정답(gold)은 주어지지 않습니다.
# =====================================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
import os
import re

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# ----------------------------- ① 설정 (knobs) -----------------------------
BASE_URL    = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
MODEL       = "qwen3:0.6b"   # 서빙 중인 모델 태그. (과제/서버 지정에 맞추세요)
TEMPERATURE = 0.0            # 0.0 = 결정적. self-consistency 라면 0.7~0.9
MAX_TOKENS  = 4096           # /think 켜면 추론 토큰 때문에 크게 둠. /no_think 면 작아도 됨
SEED        = 0


def make_llm(temperature=TEMPERATURE, max_tokens=MAX_TOKENS, seed=SEED):
    return ChatOllama(
        base_url=BASE_URL,
        model=MODEL,
        temperature=temperature,
        num_predict=max_tokens,
        seed=seed,
    )


# ----------------------------- ② 답 파싱 -----------------------------
# 모델이 형식을 벗어나도 유효한 옵션 문자를 건지도록 단계적으로 시도.
def extract_letter(text, n_options):
    valid = set(LETTERS[:n_options])
    t = str(text or "").strip().upper()

    if len(t) == 1 and t in valid:
        return t
    for marker in ["FINAL ANSWER:", "ANSWER:", "OPTION:"]:
        if marker in t:
            tail = t.split(marker, 1)[1].strip()
            if tail and tail[0] in valid:
                return tail[0]
    # 독립된 단일 문자만 매칭 ( '(B)', 'B.', ' B ' 등 ). 단어 속 글자 오인을 방지.
    for ch in re.findall(r"\b([A-Z])\b", t):
        if ch in valid:
            return ch
    return None


# ----------------------------- ③ 프롬프트 -----------------------------
# 첫 줄의 /no_think 를 /think 로 바꾸면 thinking 모드가 켜집니다(정확도↑·속도↓).
# 그때는 MAX_TOKENS 를 넉넉히(예: 4096) 두세요.
def build_prompt():
    return ChatPromptTemplate.from_messages([
        ("system", "You are a careful multiple-choice solver. Return the single best option letter."),
        ("human", """/no_think
Category:
{category}

Question:
{question}

Options:
{options}

Return the final answer in exactly this format:
Final Answer: <single letter>"""),
    ])


# ----------------------------- ④ 메인 풀이 -----------------------------
# 계약만 지키면 이 안의 전략은 자유롭게 바꿔도 됩니다.
def predict(example):
    question = example["question"]
    options = example["options"]
    category = example.get("category", "")
    option_text = "\n".join(f"{LETTERS[i]}. {opt}" for i, opt in enumerate(options))

    chain = build_prompt() | make_llm()
    try:
        msg = chain.invoke({"category": category, "question": question, "options": option_text})
        raw = getattr(msg, "content", str(msg))
        meta = getattr(msg, "response_metadata", {}) or {}
        done_reason = meta.get("done_reason", "")
        exception = ""
    except Exception as e:
        raw, meta, done_reason, exception = "", {"error": repr(e)}, "exception", repr(e)

    answer = extract_letter(raw, len(options))
    return {
        "answer": answer,
        "raw": raw,
        "artifacts": {
            "mode": "my_solver",
            "model": MODEL,
            "done_reason": done_reason,
            "exception": exception,
        },
    }


# ----------------------------- 확장 아이디어 -----------------------------
# - self-consistency: predict 안에서 make_llm(seed=i, temperature=0.8) 로 여러 번
#   호출하고, extract_letter 결과를 다수결(Counter)로 합치기.
# - reflexion: 1차 답 -> "왜 틀릴 수 있나" 자기비판 -> 2차 답 (다음 안내의 부록 참고).
# - 프롬프트 CoT: /no_think 를 /think 로 바꾸고 "Think in 2 short steps" 추가.
'''
(WORK_DIR / 'solver_template.py').write_text(solver_template, encoding='utf-8')
mine = WORK_DIR / 'solver_mine.py'
if not mine.exists():
    mine.write_text(solver_template, encoding='utf-8')
    print('created solver_mine.py — 이 파일을 편집하세요')
else:
    print('solver_mine.py 이미 있음 — 기존 편집을 유지합니다')
print('reference written: solver_template.py')


created solver_mine.py — 이 파일을 편집하세요
reference written: solver_template.py


## 7. 내 솔버 평가하기

`eval.py` 가 MMLU-Pro(카테고리 stratified, seed 고정) 부분집합에 대해 여러분의 `predict` 를 호출하고 채점합니다.

한 줄 명령(개념): `python eval.py --submission solver_mine.py --output results_mine.csv --summary-output summary_mine.json [--limit N]`

**summary.json 읽는 법**
- `n` 문항 수 · `correct` 정답 수 · `accuracy` = correct/n · `score` = accuracy×100
- `invalid` 답을 못 뽑은 문항 수(파싱 실패/예외 → 오답 처리). 이 값이 크면 **프롬프트나 파서를 먼저 손보세요.**
- `avg_elapsed_sec` 문항당 평균 시간(시간예산 감).

아래 두 셀: 먼저 `--limit 2` 로 빠르게, 그다음 전체로.

In [10]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

# 내 솔버 빠른 점검 (앞 2문항)
python eval.py \
  --submission solver_mine.py \
  --output results_mine_2.csv \
  --summary-output summary_mine_2.json \
  --limit 2

cat summary_mine_2.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 2 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 50.0,
  "invalid": 0,
  "avg_elapsed_sec": 2.5391610860824585,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry": 2,
    "computer science": 2,
    "economics": 2,
    "engineering": 2,
    "health": 2,
    "history": 2,
    "law": 2,
    "math": 2,
    "philosophy": 2,
    "physics": 2,
    "psychology": 2
  }
}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 

100%|██████████| 2/2 [00:05<00:00,  2.54s/it]


In [11]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

# 내 솔버 전체 평가 (기본 26문항)
python eval.py \
  --submission solver_mine.py \
  --output results_mine_26.csv \
  --summary-output summary_mine_26.json

cat summary_mine_26.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 26 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 26,
  "correct": 9,
  "accuracy": 0.34615384615384615,
  "score": 34.62,
  "invalid": 4,
  "avg_elapsed_sec": 8.051047591062693,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155,
    94,
    265,
    4066,
    3939,
    10231,
    10391,
    6955,
    6826,
    11574,
    11330,
    6339,
    6370,
    4722,
    4866,
    1184,
    1571,
    8279,
    8545,
    10987,
    10666,
    9054,
    9962,
    2306,
    2010
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry"

100%|██████████| 26/26 [03:29<00:00,  8.05s/it]


## 7b. (참고) 베이스라인 예시 빠른 실행 — 비교용

아래는 *제공된* 예시 솔버(`solver_single.py`, `solver_self_consistency.py`)를 비교용으로 돌립니다. 여러분 솔버 평가는 위 **7. 내 솔버 평가하기** 를 쓰세요.

In [12]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

python eval.py \
  --submission solver_single.py \
  --output results_single_2.csv \
  --summary-output summary_single_2.json \
  --limit 2

cat summary_single_2.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 2 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 2,
  "correct": 0,
  "accuracy": 0.0,
  "score": 0.0,
  "invalid": 0,
  "avg_elapsed_sec": 5.528513789176941,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry": 2,
    "computer science": 2,
    "economics": 2,
    "engineering": 2,
    "health": 2,
    "history": 2,
    "law": 2,
    "math": 2,
    "philosophy": 2,
    "physics": 2,
    "psychology": 2
  }
}
{
  "n": 2,
  "correct": 0,
  "accuracy": 0.0,
  "score": 0.

100%|██████████| 2/2 [00:11<00:00,  5.53s/it]


In [13]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

python eval.py \
  --submission solver_self_consistency.py \
  --output results_sc_2.csv \
  --summary-output summary_sc_2.json \
  --limit 2

cat summary_sc_2.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 2 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 50.0,
  "invalid": 0,
  "avg_elapsed_sec": 15.590676665306091,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry": 2,
    "computer science": 2,
    "economics": 2,
    "engineering": 2,
    "health": 2,
    "history": 2,
    "law": 2,
    "math": 2,
    "philosophy": 2,
    "physics": 2,
    "psychology": 2
  }
}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 

100%|██████████| 2/2 [00:31<00:00, 15.59s/it]


## 8. (참고) 베이스라인 예시 전체 실행 — 비교용

In [14]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

python eval.py \
  --submission solver_single.py \
  --output results_single_26.csv \
  --summary-output summary_single_26.json

cat summary_single_26.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 26 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 26,
  "correct": 7,
  "accuracy": 0.2692307692307692,
  "score": 26.92,
  "invalid": 4,
  "avg_elapsed_sec": 7.498974882639372,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155,
    94,
    265,
    4066,
    3939,
    10231,
    10391,
    6955,
    6826,
    11574,
    11330,
    6339,
    6370,
    4722,
    4866,
    1184,
    1571,
    8279,
    8545,
    10987,
    10666,
    9054,
    9962,
    2306,
    2010
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry":

100%|██████████| 26/26 [03:14<00:00,  7.50s/it]


In [15]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

python eval.py \
  --submission solver_self_consistency.py \
  --output results_sc_26.csv \
  --summary-output summary_sc_26.json

cat summary_sc_26.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 26 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 26,
  "correct": 7,
  "accuracy": 0.2692307692307692,
  "score": 26.92,
  "invalid": 0,
  "avg_elapsed_sec": 33.93510359067183,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155,
    94,
    265,
    4066,
    3939,
    10231,
    10391,
    6955,
    6826,
    11574,
    11330,
    6339,
    6370,
    4722,
    4866,
    1184,
    1571,
    8279,
    8545,
    10987,
    10666,
    9054,
    9962,
    2306,
    2010
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry":

100%|██████████| 26/26 [14:42<00:00, 33.94s/it]


### 📊 결과 읽고 디버깅하기

아래 셀들(요약 비교 · 결과 CSV 열람 · self-consistency 표 펼치기 · 원문 출력)은 **여러분 솔버 결과에도 그대로** 쓸 수 있습니다. 파일명을 `summary_mine_*.json` / `results_mine_*.csv` 로 바꾸기만 하면 됩니다.

- 틀린 문항: CSV에서 `correct=False` 행의 `raw_pred`(모델 원문)와 `artifact_json`(여러분이 남긴 디버그 정보)을 보고 원인을 찾으세요.
- `pred` 가 빈칸이면 **답 추출 실패** → `extract_letter` 나 출력 형식 지시를 고치세요.
- `elapsed_sec` 가 너무 길면 `/think`·`num_predict`·샘플 수를 조절해 시간을 줄이세요.

## 9. Compare summaries

In [16]:
import json
import pandas as pd
from IPython.display import display

summary_files = [
    "summary_single_2.json",
    "summary_sc_2.json",
    "summary_single_26.json",
    "summary_sc_26.json",
]

rows = []
for filename in summary_files:
    path = WORK_DIR / filename
    if not path.exists():
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "run": filename.replace("summary_", "").replace(".json", ""),
        "n": data.get("n"),
        "correct": data.get("correct"),
        "accuracy": data.get("accuracy"),
        "score": data.get("score"),
        "invalid": data.get("invalid"),
        "avg_elapsed_sec": data.get("avg_elapsed_sec"),
    })

display(pd.DataFrame(rows))

,run,n,correct,accuracy,score,invalid,avg_elapsed_sec
0,single_2,2,0,0.000000,0.00,0,5.528514
1,sc_2,2,1,0.500000,50.00,0,15.590677
2,single_26,26,7,0.269231,26.92,4,7.498975
3,sc_26,26,7,0.269231,26.92,0,33.935104


## 10. Inspect result CSV

In [17]:
import pandas as pd
from IPython.display import display

# Change this to inspect another run:
#   results_single_2.csv
#   results_sc_2.csv
#   results_single_26.csv
#   results_sc_26.csv
result_path = WORK_DIR / "results_sc_2.csv"

df = pd.read_csv(result_path)

cols = [
    c for c in [
        "local_idx",
        "dataset_idx",
        "category",
        "pred",
        "gold",
        "correct",
        "elapsed_sec",
        "raw_pred",
        "artifact_json",
        "error",
        "question",
    ]
    if c in df.columns
]

display(df[cols])

,local_idx,dataset_idx,category,pred,gold,correct,elapsed_sec,raw_pred,artifact_json,error,question
0,0,3301,biology,H,H,True,15.185735,--- SAMPLE 0 | seed=0 | letter=H | status=ok |...,"{""mode"": ""self_consistency"", ""model"": ""qwen3:0...",NaN,If a living frog muscle is dissected out from ...
1,1,3155,biology,C,F,False,15.995618,--- SAMPLE 0 | seed=0 | letter=C | status=ok |...,"{""mode"": ""self_consistency"", ""model"": ""qwen3:0...",NaN,Electrons are located at definite energy level...


## 11. Expand self-consistency artifacts

In [18]:
import json
import pandas as pd
from IPython.display import display

# This cell works best with results_sc_2.csv or results_sc_26.csv.
# It silently skips non-self-consistency rows.
artifact_rows = []
vote_rows = []

for _, row in df.iterrows():
    artifact_text = row.get("artifact_json", "")

    if not isinstance(artifact_text, str) or not artifact_text.strip():
        continue

    artifact = json.loads(artifact_text)

    if artifact.get("mode") != "self_consistency":
        continue

    vote_rows.append({
        "local_idx": row.get("local_idx", ""),
        "category": row.get("category", ""),
        "gold": row.get("gold", ""),
        "final_pred": row.get("pred", ""),
        "correct": row.get("correct", ""),
        "n_samples": artifact.get("n_samples", ""),
        "votes": artifact.get("votes", {}),
        "status": artifact.get("status", ""),
    })

    for sample in artifact.get("samples", []):
        artifact_rows.append({
            "local_idx": row.get("local_idx", ""),
            "gold": row.get("gold", ""),
            "final_pred": row.get("pred", ""),
            "votes": artifact.get("votes", {}),
            "sample_idx": sample.get("sample_idx", ""),
            "seed": sample.get("seed", ""),
            "strategy": sample.get("strategy", ""),
            "sample_letter": sample.get("letter", ""),
            "sample_status": sample.get("status", ""),
            "done_reason": sample.get("done_reason", ""),
            "raw_generation": sample.get("raw_generation", ""),
        })

print("Vote summary")
display(pd.DataFrame(vote_rows))

print("Per-sample generations")
display(pd.DataFrame(artifact_rows))

Vote summary


,local_idx,category,gold,final_pred,correct,n_samples,votes,status
0,0,biology,H,H,True,5,"{'H': 4, 'D': 1}",ok
1,1,biology,F,C,False,5,{'C': 5},ok


Per-sample generations


,local_idx,gold,final_pred,votes,sample_idx,seed,strategy,sample_letter,sample_status,done_reason,raw_generation
0,0,H,H,"{'H': 4, 'D': 1}",0,0,Solve directly and choose the best answer.,H,ok,stop,Final Answer: H
1,0,H,H,"{'H': 4, 'D': 1}",1,1,"Eliminate clearly wrong options first, then ch...",H,ok,stop,Final Answer: H
2,0,H,H,"{'H': 4, 'D': 1}",2,2,Check each option briefly and choose the most ...,H,ok,stop,Final Answer: H
3,0,H,H,"{'H': 4, 'D': 1}",3,3,Look for traps or misleading options before ch...,D,ok,stop,Final Answer: D
4,0,H,H,"{'H': 4, 'D': 1}",4,4,Use concise domain reasoning and avoid overthi...,H,ok,stop,Final Answer: H
5,1,F,C,{'C': 5},0,0,Solve directly and choose the best answer.,C,ok,stop,Final Answer: C
6,1,F,C,{'C': 5},1,1,"Eliminate clearly wrong options first, then ch...",C,ok,stop,Final Answer: C
7,1,F,C,{'C': 5},2,2,Check each option briefly and choose the most ...,C,ok,stop,Final Answer: C
8,1,F,C,{'C': 5},3,3,Look for traps or misleading options before ch...,C,ok,stop,Final Answer: C
9,1,F,C,{'C': 5},4,4,Use concise domain reasoning and avoid overthi...,C,ok,stop,Final Answer: C


## 12. Print raw generations for the current CSV

In [19]:
for _, row in df.iterrows():
    print("=" * 120)
    print(f"local_idx: {row.get('local_idx', '')}")
    print(f"category: {row.get('category', '')}")
    print(f"pred: {row.get('pred', '')}")
    print(f"gold: {row.get('gold', '')}")
    print(f"correct: {row.get('correct', '')}")
    print("-" * 120)
    print(row.get("raw_pred", ""))
    print()

local_idx: 0
category: biology
pred: H
gold: H
correct: True
------------------------------------------------------------------------------------------------------------------------
--- SAMPLE 0 | seed=0 | letter=H | status=ok | done_reason=stop ---
Final Answer: H

--- SAMPLE 1 | seed=1 | letter=H | status=ok | done_reason=stop ---
Final Answer: H

--- SAMPLE 2 | seed=2 | letter=H | status=ok | done_reason=stop ---
Final Answer: H

--- SAMPLE 3 | seed=3 | letter=D | status=ok | done_reason=stop ---
Final Answer: D

--- SAMPLE 4 | seed=4 | letter=H | status=ok | done_reason=stop ---
Final Answer: H

local_idx: 1
category: biology
pred: C
gold: F
correct: False
------------------------------------------------------------------------------------------------------------------------
--- SAMPLE 0 | seed=0 | letter=C | status=ok | done_reason=stop ---
Final Answer: C

--- SAMPLE 1 | seed=1 | letter=C | status=ok | done_reason=stop ---
Final Answer: C

--- SAMPLE 2 | seed=2 | letter=C | statu

## ▶️ 다음 단계: 베이스라인을 넘어서

- **프롬프트 CoT** — `/think` 로 짧은 단계 추론 후 답을 고르기.
- **Self-consistency** — 여러 샘플의 다수결 (`solver_self_consistency.py` 참고).
- **Reflexion / 자기수정, DSPy 프롬프트 최적화** — 다음 안내(부록)에서 실행 가능한 스켈레톤과 함께 다룹니다.

어떤 기법이든 결국 **`predict` 의 내부 구현만 바뀌고 채점 계약은 그대로**입니다. 그래서 한 가지를 바꾸면 즉시 위 평가 셀로 점수를 비교할 수 있습니다.

# 부록: 베이스라인을 넘어서는 방법

아래 두 기법은 **선택 확장**입니다. 각각 참고용 예시 파일을 만들어 두니, 마음에 들면 `solver_mine.py` 로 **복사해서** 발전시키세요.
어떤 기법이든 결국 `predict(example)` 의 내부만 바뀌고 채점 계약은 그대로입니다.

## 부록 A. Reflexion (자기수정)

**아이디어**: 한 번에 답하지 말고, ① 1차 답 → ② 그 답을 스스로 비판("어디가 틀릴 수 있나?") → ③ 교정된 최종 답, 의 순서로 두 번 생각하게 합니다. 모델이 첫 답에서 흔들렸을 때 되짚을 기회를 주는 방식입니다.

- **장점**: 함정·헷갈리는 보기에서 자기교정 여지가 생깁니다.
- **비용**: 문항당 LLM 호출이 2배 → 느려집니다. (latency 예산 확인)
- **주의**: 단발 객관식은 되짚을 신호가 약할 수 있어, 항상 정확도가 오르지는 않습니다. **효과가 없었다면 그 사실과 원인 분석도 좋은 보고입니다.**

아래 셀이 `solver_reflexion.py`(참고 예시)를 만들고, 2문항으로 빠르게 평가합니다.

In [20]:
from pathlib import Path
solver_reflexion_src = r'''
# =====================================================================
# solver_reflexion.py  —  Reflexion(자기수정) 예시 솔버
# 편집해서 제출하려면 이 파일을 solver_mine.py 로 "복사"한 뒤 고치세요.
# 계약: predict(example) -> {"answer","raw","artifacts"}
# =====================================================================
from langchain_ollama import ChatOllama
import os
import re

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
MODEL = "qwen3:0.6b"
MAX_TOKENS = 4096


def make_llm(temperature=0.0, seed=0):
    return ChatOllama(base_url=BASE_URL, model=MODEL,
                      temperature=temperature, num_predict=MAX_TOKENS, seed=seed)


def extract_letter(text, n_options):
    valid = set(LETTERS[:n_options])
    t = str(text or "").strip().upper()
    if len(t) == 1 and t in valid:
        return t
    for marker in ["FINAL ANSWER:", "ANSWER:", "OPTION:"]:
        if marker in t:
            tail = t.split(marker, 1)[1].strip()
            if tail and tail[0] in valid:
                return tail[0]
    for ch in re.findall(r"\b([A-Z])\b", t):
        if ch in valid:
            return ch
    return None


def _ask(messages, seed=0, temperature=0.0):
    try:
        msg = make_llm(temperature=temperature, seed=seed).invoke(messages)
        return getattr(msg, "content", str(msg))
    except Exception:
        return ""


def predict(example):
    q = example["question"]
    options = example["options"]
    category = example.get("category", "")
    opt = "\n".join(f"{LETTERS[i]}. {o}" for i, o in enumerate(options))
    n = len(options)

    # 1차 시도
    first = _ask([
        ("system", "You are a careful multiple-choice solver."),
        ("human", """/no_think
Category: {cat}
Question:
{q}
Options:
{opt}
Reason in 1-2 short sentences, then output 'Final Answer: <letter>'.""".format(cat=category, q=q, opt=opt)),
    ])
    a1 = extract_letter(first, n)

    # 자기비판 + 재시도
    second = _ask([
        ("system", "You critique and, if needed, correct a previous multiple-choice answer."),
        ("human", """/no_think
Question:
{q}
Options:
{opt}

A previous attempt chose: {a1}
Its reasoning:
{first}

Point out any mistake in one short sentence, then give the corrected
'Final Answer: <letter>'.""".format(q=q, opt=opt, a1=a1, first=first)),
    ], seed=1)
    a2 = extract_letter(second, n)

    final = a2 or a1
    return {
        "answer": final,
        "raw": "[attempt1]\n" + first + "\n\n[reflection]\n" + second,
        "artifacts": {"mode": "reflexion", "model": MODEL,
                      "attempt1": a1, "attempt2": a2, "final": final},
    }
'''
(WORK_DIR / 'solver_reflexion.py').write_text(solver_reflexion_src, encoding='utf-8')
print('wrote reference:', 'solver_reflexion.py', '(편집하려면 solver_mine.py 로 복사하세요)')


wrote reference: solver_reflexion.py (편집하려면 solver_mine.py 로 복사하세요)


In [21]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

# Reflexion 예시 빠른 평가 (호출이 2배라 느립니다 -> 우선 2문항)
python eval.py \
  --submission solver_reflexion.py \
  --output results_reflexion_2.csv \
  --summary-output summary_reflexion_2.json \
  --limit 2

cat summary_reflexion_2.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 2 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 50.0,
  "invalid": 0,
  "avg_elapsed_sec": 5.561010241508484,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry": 2,
    "computer science": 2,
    "economics": 2,
    "engineering": 2,
    "health": 2,
    "history": 2,
    "law": 2,
    "math": 2,
    "philosophy": 2,
    "physics": 2,
    "psychology": 2
  }
}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 5

100%|██████████| 2/2 [00:11<00:00,  5.56s/it]


## 부록 B. DSPy 프롬프트 최적화

**아이디어**: 프롬프트를 손으로 깎는 대신, **입출력 시그니처**(`question, options -> answer`)만 선언하고 DSPy가 프롬프트·few-shot 예시를 *데이터로 최적화*하게 합니다.

- `dspy.ChainOfThought(시그니처)` 만으로 zero-shot CoT 베이스라인이 됩니다 (아래 예시).
- **진짜 장점은 optimizer**: 라벨이 있는 dev set(`challenge_dev.json`)으로 `BootstrapFewShot`/`MIPROv2` 가 좋은 few-shot 데모와 지시문을 자동 선택합니다. (예시 파일 하단에 주석으로 골격 포함)
- **비용/주의**: 최적화는 LLM 호출이 많이 듭니다. 또 0.6B 같은 작은 모델은 DSPy의 구조화 출력 파싱이 가끔 실패할 수 있어 `extract_letter` 폴백을 둡니다.

먼저 DSPy를 설치하고, `solver_dspy.py`(참고 예시)를 만든 뒤 2문항으로 평가합니다.

In [22]:
!pip -q install dspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 27.7 MB/s eta 0:00:00


In [23]:
from pathlib import Path
solver_dspy_src = r'''
# =====================================================================
# solver_dspy.py  —  DSPy 프롬프트 최적화 예시 솔버
# 편집해서 제출하려면 이 파일을 solver_mine.py 로 "복사"한 뒤 고치세요.
# 계약: predict(example) -> {"answer","raw","artifacts"}
# 사전 설치 필요: pip install dspy
# =====================================================================
import re
import dspy

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
BASE_URL = "http://127.0.0.1:11434"
MODEL = "qwen3:0.6b"

# DSPy LM 설정 (모듈 import 시 1회). Ollama 는 ollama_chat/<태그> 형식.
_lm = dspy.LM("ollama_chat/" + MODEL, api_base=BASE_URL, api_key="",
              temperature=0.0, max_tokens=4096)
dspy.configure(lm=_lm)


class MCQA(dspy.Signature):
    """Answer a multiple-choice question. Output only the letter of the best option."""
    question: str = dspy.InputField()
    options: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="single option letter, e.g. A")


# zero-shot Chain-of-Thought 프로그램 (최적화 전 베이스라인)
_program = dspy.ChainOfThought(MCQA)


def extract_letter(text, n_options):
    valid = set(LETTERS[:n_options])
    t = str(text or "").strip().upper()
    if len(t) == 1 and t in valid:
        return t
    for marker in ["FINAL ANSWER:", "ANSWER:", "OPTION:"]:
        if marker in t:
            tail = t.split(marker, 1)[1].strip()
            if tail and tail[0] in valid:
                return tail[0]
    for ch in re.findall(r"\b([A-Z])\b", t):
        if ch in valid:
            return ch
    return None


def predict(example):
    q = example["question"]
    options = example["options"]
    opt = "\n".join(f"{LETTERS[i]}. {o}" for i, o in enumerate(options))
    try:
        out = _program(question="/no_think " + q, options=opt)
        raw = getattr(out, "answer", "")
        reasoning = getattr(out, "reasoning", "")
        err = ""
    except Exception as e:
        raw, reasoning, err = "", "", repr(e)
    ans = extract_letter(raw, len(options)) or extract_letter(reasoning, len(options))
    return {
        "answer": ans,
        "raw": str(reasoning) + "\nAnswer: " + str(raw),
        "artifacts": {"mode": "dspy_cot", "model": MODEL, "exception": err},
    }


# ---------------------------------------------------------------------
# (선택) 프롬프트 최적화 — DSPy 의 진짜 장점.
# 라벨이 있는 dev set(challenge_dev.json) 으로 few-shot/instruction 을 자동 튜닝한다.
# 호출이 많이 드니 작은 trainset 으로 시작하세요.
#
# from dspy.teleprompt import BootstrapFewShot
# import json
#
# def _metric(example, pred, trace=None):
#     return extract_letter(getattr(pred, "answer", ""), 10) == example.gold
#
# dev = json.load(open("/content/drive/MyDrive/nlp_hw2/challenge_dev.json"))
# trainset = [
#     dspy.Example(
#         question="/no_think " + d["question"],
#         options="\n".join(f"{LETTERS[i]}. {o}" for i, o in enumerate(d["options"])),
#         gold=d["answer"],
#     ).with_inputs("question", "options")
#     for d in dev
# ]
# optimizer = BootstrapFewShot(metric=_metric, max_bootstrapped_demos=4)
# _program = optimizer.compile(_program, trainset=trainset)   # predict() 가 이 최적화된 프로그램을 사용
'''
(WORK_DIR / 'solver_dspy.py').write_text(solver_dspy_src, encoding='utf-8')
print('wrote reference:', 'solver_dspy.py', '(편집하려면 solver_mine.py 로 복사하세요)')


wrote reference: solver_dspy.py (편집하려면 solver_mine.py 로 복사하세요)


In [24]:
%%bash -s "$WORK_DIR"
set -e
WORK_DIR="$1"
cd "$WORK_DIR"

# DSPy 예시 빠른 평가 (먼저 2문항)
python eval.py \
  --submission solver_dspy.py \
  --output results_dspy_2.csv \
  --summary-output summary_dspy_2.json \
  --limit 2

cat summary_dspy_2.json

[eval] loading dataset: TIGER-Lab/MMLU-Pro/test
[eval] selected 2 examples
[eval] category_counts={'biology': 2, 'business': 2, 'chemistry': 2, 'computer science': 2, 'economics': 2, 'engineering': 2, 'health': 2, 'history': 2, 'law': 2, 'math': 2, 'philosophy': 2, 'physics': 2, 'psychology': 2}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 50.0,
  "invalid": 0,
  "avg_elapsed_sec": 2.811492919921875,
  "dataset": "TIGER-Lab/MMLU-Pro",
  "split": "test",
  "n_per_category": 2,
  "max_categories": 0,
  "exclude_categories": [
    "other"
  ],
  "include_categories": [],
  "sample_seed": 20260630,
  "selected_indices": [
    3301,
    3155
  ],
  "category_counts": {
    "biology": 2,
    "business": 2,
    "chemistry": 2,
    "computer science": 2,
    "economics": 2,
    "engineering": 2,
    "health": 2,
    "history": 2,
    "law": 2,
    "math": 2,
    "philosophy": 2,
    "physics": 2,
    "psychology": 2
  }
}
{
  "n": 2,
  "correct": 1,
  "accuracy": 0.5,
  "score": 5

100%|██████████| 2/2 [00:05<00:00,  2.81s/it]


## 마무리

- **Reflexion**: `predict` 안에서 1차→자기비판→교정. 정확도-속도 트레이드오프를 측정해 보세요.
- **DSPy**: 프롬프트를 데이터로 최적화. zero-shot부터 시작해 optimizer로 확장.

두 방식 모두 같은 `eval.py` 로 즉시 비교됩니다. 가장 좋은 제출은 *점수*뿐 아니라, **무엇을 왜 시도했고 어떻게 측정했는지**(method 문서)가 분명한 것입니다.